In [1]:
"""
Compact DDPM (Denoising Diffusion Probabilistic Model) on MNIST.
Trains a small U-Net to predict noise, then samples images to show
generation quality as a grid saved to disk.

Run:  python diffusion_demo.py
Needs: torch, torchvision, matplotlib
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------- Diffusion process -----------------------------
class Diffusion:
    def __init__(self, timesteps=300, beta_start=1e-4, beta_end=0.02, device=device):
        self.T = timesteps
        self.beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alpha = 1. - self.beta
        self.alpha_bar = torch.cumprod(self.alpha, dim=0)

    def q_sample(self, x0, t, noise=None):
        noise = torch.randn_like(x0) if noise is None else noise
        sqrt_ab = self.alpha_bar[t].sqrt().view(-1, 1, 1, 1)
        sqrt_1mab = (1 - self.alpha_bar[t]).sqrt().view(-1, 1, 1, 1)
        return sqrt_ab * x0 + sqrt_1mab * noise, noise

    def loss(self, model, x0):
        t = torch.randint(0, self.T, (x0.size(0),), device=x0.device)
        x_t, noise = self.q_sample(x0, t)
        pred_noise = model(x_t, t)
        return F.mse_loss(pred_noise, noise)

    @torch.no_grad()
    def sample(self, model, shape, device=device):
        x = torch.randn(shape, device=device)
        for t in reversed(range(self.T)):
            t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
            beta_t, alpha_t, alpha_bar_t = self.beta[t], self.alpha[t], self.alpha_bar[t]
            pred_noise = model(x, t_batch)
            mean = (1 / alpha_t.sqrt()) * (x - (beta_t / (1 - alpha_bar_t).sqrt()) * pred_noise)
            x = mean + beta_t.sqrt() * torch.randn_like(x) if t > 0 else mean
        return x.clamp(-1, 1)

# ----------------------------- Tiny U-Net -----------------------------
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.mlp(emb)

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.act = nn.SiLU()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act(self.norm1(self.conv1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.act(self.norm2(self.conv2(h)))
        return h + self.skip(x)

class TinyUNet(nn.Module):
    def __init__(self, ch=64, time_dim=128):
        super().__init__()
        self.time_emb = TimeEmbedding(time_dim)
        self.in_conv = nn.Conv2d(1, ch, 3, padding=1)
        self.down1 = Block(ch, ch, time_dim)
        self.down2 = Block(ch, ch * 2, time_dim)
        self.pool = nn.AvgPool2d(2)
        self.mid = Block(ch * 2, ch * 2, time_dim)
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.up1 = Block(ch * 2 + ch * 2, ch, time_dim)
        self.up2 = Block(ch + ch, ch, time_dim)
        self.out_conv = nn.Conv2d(ch, 1, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_emb(t)
        x0 = self.in_conv(x)
        d1 = self.down1(x0, t_emb)
        d2 = self.down2(self.pool(d1), t_emb)
        m = self.mid(self.pool(d2), t_emb)
        u1 = self.up1(torch.cat([self.up(m), d2], dim=1), t_emb)
        u2 = self.up2(torch.cat([self.up(u1), d1], dim=1), t_emb)
        return self.out_conv(u2)

# ----------------------------- Train + sample -----------------------------

tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=tf)
loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, drop_last=True)

model = TinyUNet().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
diffusion = Diffusion(timesteps=300, device=device)

epochs = 5  # increase for better quality (e.g. 20-50)
for epoch in range(epochs):
    running = 0.0
    for x, _ in loader:
        x = x.to(device)
        loss = diffusion.loss(model, x)
        opt.zero_grad()
        loss.backward()
        opt.step()
        running += loss.item()
    print(f"epoch {epoch+1}/{epochs}  loss={running/len(loader):.4f}")

    # Sample a grid after every epoch to watch quality improve
    model.eval()
    samples = diffusion.sample(model, (16, 1, 28, 28), device=device)
    samples = (samples + 1) / 2  # back to [0,1]
    grid = make_grid(samples, nrow=4)
    save_image(grid, f"samples_epoch{epoch+1}.png")
    model.train()

print("Done. Check samples_epoch*.png to see generation quality improve over training.")



100%|██████████████████████████████████████| 9.91M/9.91M [00:03<00:00, 2.50MB/s]
100%|███████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 129kB/s]
100%|██████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 2.08MB/s]
100%|██████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 1.89MB/s]


KeyboardInterrupt: 